# HSI vs CPI: Full EDA Suite

This notebook builds an end-to-end exploratory analysis pipeline to detect patterns between the Hang Seng Index (HSI) and Hong Kong CPI series.

## What this covers
- Data loading and robust cleaning
- CPI parsing from a multi-row header file
- Daily-to-monthly HSI feature engineering
- Temporal alignment and merge
- Trend and distribution visualizations
- Correlation, lagged correlation, and rolling correlation
- Regime and seasonality checks
- OLS regression diagnostics
- Granger causality tests


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import yfinance as yf

from scipy import stats
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
import statsmodels.api as sm

HSI_TICKER = '^HSI'
HSI_START = '2000-01-01'
DATA_CPI = 'HK_CPI_data'


## 1) Load raw data

In [ ]:
hsi_raw = yf.download(HSI_TICKER, start=HSI_START, progress=False, auto_adjust=False)
if hsi_raw.empty:
    raise ValueError(f'No HSI data returned from yfinance for {HSI_TICKER}.')
if isinstance(hsi_raw.columns, pd.MultiIndex):
    hsi_raw.columns = hsi_raw.columns.get_level_values(0)
hsi_raw = hsi_raw.reset_index()
cpi_raw = pd.read_csv(DATA_CPI, header=None)

print('HSI raw shape:', hsi_raw.shape)
print('CPI raw shape:', cpi_raw.shape)

display(hsi_raw.head())
display(cpi_raw.head(10))


## 2) Clean CPI table (multi-row header)

The CPI file includes title/meta rows and grouped headers. We detect the true `Year, Month` row and rebuild readable column names.


In [ ]:
raw = cpi_raw.copy()

#--------------------------- cleaning the file ----------------------------

# Locate the true header row by finding where col0 == "Year" and col1 == "Month".
# This is more robust than hard-coding a row number because source files may shift.
hdr_idx = raw.index[
    (raw.iloc[:, 0].astype(str).str.strip().eq("Year"))
    & (raw.iloc[:, 1].astype(str).str.strip().eq("Month"))
][0]

# The table uses a 2-row grouped header above the "Year/Month" row:
# - h1: high-level group (e.g., CPI type)
# - h2: metric (e.g., Index, YoY change, MoM change)
h1 = raw.iloc[hdr_idx - 2].ffill()  # forward-fill group names across merged-like cells
h2 = raw.iloc[hdr_idx - 1]

# Build clean, unique column names by combining h1 and h2.
new_cols = []
seen = {}  # tracks duplicates to enforce uniqueness

for i in range(raw.shape[1]):
    if i == 0:
        col = "Year"
    elif i == 1:
        col = "Month"
    else:
        a = str(h1[i]).strip() if pd.notna(h1[i]) else ""
        b = str(h2[i]).strip() if pd.notna(h2[i]) else ""

        # Prefer "Group - Metric"; fall back to whichever exists; else generated name.
        col = f"{a} - {b}" if a and b else (a or b or f"col_{i}")

        # Normalize symbols/text to avoid awkward column names in downstream code.
        col = col.replace("%", "pct")

        # Remove non-breaking spaces and collapse extra whitespace to prevent hidden
        # near-duplicates (e.g., visually same but technically different names).
        col = col.replace("\xa0", " ")
        col = " ".join(col.split())

    # Ensure uniqueness in case normalization produces duplicate names.
    if col in seen:
        seen[col] += 1
        col = f"{col}_{seen[col]}"
    else:
        seen[col] = 0

    new_cols.append(col)

# Extract data rows only (everything below the header marker row) and assign new columns.
cpi_df = raw.iloc[hdr_idx + 1 :].copy()
cpi_df.columns = new_cols

# Clean and convert all value columns (exclude Year/Month at positions 0 and 1).
for col in cpi_df.columns[2:]:
    cpi_df[col] = (
        cpi_df[col]
        .astype(str)
        # Remove bracketed footnotes like "[1]" that break numeric parsing.
        .str.replace(r"\s*\[.*?\]", "", regex=True)
        # Standardize missing-value markers before numeric conversion.
        .replace({"N.A.": pd.NA, "nan": pd.NA, "": pd.NA})
    )
    # Coerce invalid strings to NaN so numeric operations work safely.
    cpi_df[col] = pd.to_numeric(cpi_df[col], errors="coerce")

# Convert Year/Month to year and month numbers only (nullable integers).
year_dt = pd.to_datetime(
    cpi_df["Year"].astype("string").str.strip(),
    format="%Y",
    errors="coerce",
)

month_str = cpi_df["Month"].astype("string").str.strip()
month_num = pd.to_numeric(month_str, errors="coerce")
month_num = month_num.fillna(pd.to_datetime(month_str, format="%b", errors="coerce").dt.month)
month_num = month_num.fillna(pd.to_datetime(month_str, format="%B", errors="coerce").dt.month)

cpi_df["Year"] = year_dt.dt.year.astype("Int64")
cpi_df["Month"] = month_num.astype("Int64")

# Keep only rows with a valid year (drops footer/notes rows).
cpi_df = cpi_df[cpi_df["Year"].notna()].reset_index(drop=True)

# Show only Year and Month preview.
display(cpi_df[["Year", "Month"]].head())

# Quick sanity check of the cleaned output.
display(cpi_df.head())

# Shorten cpi_df column names
short_name_map = {
    "Consumer Price Index": "CPI",
    "Composite CPI": "CCPI",
    "CPI (A)": "CPI_A",
    "CPI (B)": "CPI_B",
    "CPI (C)": "CPI_C",
    "Year-on-year pct change": "yoy_pct",
    "Month-to-month pct change": "mom_pct",
    "Index": "idx",
}

cpi_df = cpi_df.rename(
    columns=lambda col: col if col in ["Year", "Month"] else (
        col.replace("Consumer Price Index", short_name_map["Consumer Price Index"])
           .replace("Composite CPI", short_name_map["Composite CPI"])
           .replace("CPI (A)", short_name_map["CPI (A)"])
           .replace("CPI (B)", short_name_map["CPI (B)"])
           .replace("CPI (C)", short_name_map["CPI (C)"])
           .replace("Year-on-year pct change", short_name_map["Year-on-year pct change"])
           .replace("Month-to-month pct change", short_name_map["Month-to-month pct change"])
           .replace("Index", short_name_map["Index"])
           .replace(" - ", "_")
           .replace(" ", "")
    )
)

display(cpi_df.columns)

# Use monthly records only (annual rows have Month = <NA>)
monthly = cpi_df[cpi_df["Month"].notna()].copy()
monthly["Date"] = pd.to_datetime(
    dict(year=monthly["Year"].astype(int), month=monthly["Month"].astype(int), day=1)
) + pd.offsets.MonthEnd(0)
monthly = monthly.sort_values("Date").reset_index(drop=True)

# Keep downstream variable name unchanged for the rest of the notebook
cpi = monthly.copy()
print('Clean CPI monthly shape:', cpi.shape)
print('Date range:', cpi['Date'].min().date(), '->', cpi['Date'].max().date())
display(cpi.head())

## 3) Clean HSI and create monthly features

In [ ]:
hsi = hsi_raw.copy()

# Parse date and numeric fields
hsi['Date'] = pd.to_datetime(hsi['Date'], errors='coerce')
for c in ['Close', 'High', 'Low', 'Open', 'Volume']:
    hsi[c] = pd.to_numeric(hsi[c], errors='coerce')

hsi = hsi.dropna(subset=['Date', 'Close']).sort_values('Date').reset_index(drop=True)

# Explicit filter requested: 2000 onwards
hsi = hsi[hsi['Date'] >= pd.Timestamp('2000-01-01')].copy()

# Daily return and volatility proxy
hsi['Daily_Return'] = hsi['Close'].pct_change()
hsi['Log_Return'] = np.log(hsi['Close'] / hsi['Close'].shift(1))

# Monthly aggregation
monthly_hsi = (
    hsi.set_index('Date')
       .resample('ME')
       .agg(
           HSI_Close=('Close', 'last'),
           HSI_Open=('Open', 'first'),
           HSI_High=('High', 'max'),
           HSI_Low=('Low', 'min'),
           HSI_DailyRet_Mean=('Daily_Return', 'mean'),
           HSI_DailyRet_Std=('Daily_Return', 'std'),
           HSI_LogRet_Sum=('Log_Return', 'sum')
       )
       .reset_index()
)

monthly_hsi['HSI_Monthly_Return'] = monthly_hsi['HSI_Close'].pct_change()
monthly_hsi['HSI_YoY_Return'] = monthly_hsi['HSI_Close'].pct_change(12)

print('Monthly HSI shape:', monthly_hsi.shape)
print('Date range:', monthly_hsi['Date'].min().date(), '->', monthly_hsi['Date'].max().date())
display(monthly_hsi.head())


## 4) Select CPI features and merge with HSI

In [ ]:
# Pick commonly used CPI signals (supports both long source names and shortened names)
def pick_col(columns, all_keywords=None, any_keywords=None):
    all_keywords = all_keywords or []
    any_keywords = any_keywords or []
    for col in columns:
        col_l = col.lower()
        if all(k.lower() in col_l for k in all_keywords):
            if not any_keywords or any(k.lower() in col_l for k in any_keywords):
                return col
    return None

cpi_cols = cpi.columns.tolist()
cpi_groups = ['CCPI', 'CPI_A', 'CPI_B', 'CPI_C']

selected = ['Date']
rename_map = {}

for grp in cpi_groups:
    idx_col = pick_col(cpi_cols, all_keywords=[grp], any_keywords=['idx', 'index'])
    yoy_col = pick_col(cpi_cols, all_keywords=[grp], any_keywords=['yoy_pct', 'year-on-year'])
    mom_col = pick_col(cpi_cols, all_keywords=[grp], any_keywords=['mom_pct', 'month-to-month'])

    if grp == 'CCPI':
        idx_col = idx_col or pick_col(cpi_cols, all_keywords=['composite consumer price index'], any_keywords=['index'])
        yoy_col = yoy_col or pick_col(cpi_cols, all_keywords=['composite consumer price index'], any_keywords=['year-on-year'])
        mom_col = mom_col or pick_col(cpi_cols, all_keywords=['composite consumer price index'], any_keywords=['month-to-month'])

    if idx_col is not None:
        selected.append(idx_col)
        rename_map[idx_col] = f'{grp}_Index'
    if yoy_col is not None:
        selected.append(yoy_col)
        rename_map[yoy_col] = f'{grp}_YoY_Pct'
    if mom_col is not None:
        selected.append(mom_col)
        rename_map[mom_col] = f'{grp}_MoM_Pct'

selected = ['Date'] + list(dict.fromkeys([c for c in selected if c != 'Date']))
cpi_sel = cpi[selected].copy().rename(columns=rename_map)

# Build missing YoY/MoM from index if needed
for grp in cpi_groups:
    idx_name = f'{grp}_Index'
    yoy_name = f'{grp}_YoY_Pct'
    mom_name = f'{grp}_MoM_Pct'
    if idx_name in cpi_sel.columns and yoy_name not in cpi_sel.columns:
        cpi_sel[yoy_name] = cpi_sel[idx_name].pct_change(12) * 100
    if idx_name in cpi_sel.columns and mom_name not in cpi_sel.columns:
        cpi_sel[mom_name] = cpi_sel[idx_name].pct_change(1) * 100

# Backward-compatible aliases used by older cells
if 'CCPI_Index' in cpi_sel.columns and 'CPI_Index' not in cpi_sel.columns:
    cpi_sel['CPI_Index'] = cpi_sel['CCPI_Index']
if 'CCPI_YoY_Pct' in cpi_sel.columns and 'CPI_YoY_Pct' not in cpi_sel.columns:
    cpi_sel['CPI_YoY_Pct'] = cpi_sel['CCPI_YoY_Pct']
if 'CCPI_MoM_Pct' in cpi_sel.columns and 'CPI_MoM_Pct' not in cpi_sel.columns:
    cpi_sel['CPI_MoM_Pct'] = cpi_sel['CCPI_MoM_Pct']

df = monthly_hsi.merge(cpi_sel, on='Date', how='inner').sort_values('Date').reset_index(drop=True)

# Final guard: analysis period starts from 2000-01-01
df = df[df['Date'] >= pd.Timestamp('2000-01-01')].reset_index(drop=True)

print('Merged dataset shape:', df.shape)
print('Merged date range:', df['Date'].min().date(), '->', df['Date'].max().date())
print('CPI series available:', [c for c in df.columns if c.endswith('_Index') or c.endswith('_YoY_Pct') or c.endswith('_MoM_Pct')])
display(df.head())
display(df.tail())

## 5) Data quality snapshot

In [ ]:
summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2),
    'n_unique': df.nunique()
})

display(summary)
display(df.describe(include='all').T)


## 6) Trend visualization (level + standardized)

Standardized plots help compare co-movement despite different scales.


In [ ]:
cpi_index_cols = [c for c in ['CCPI_Index', 'CPI_A_Index', 'CPI_B_Index', 'CPI_C_Index'] if c in df.columns]

fig = go.Figure()
fig.add_trace(go.Scatter(x=df['Date'], y=df['HSI_Close'], mode='lines', name='HSI_Close', yaxis='y1'))
for col in cpi_index_cols:
    fig.add_trace(go.Scatter(x=df['Date'], y=df[col], mode='lines', name=col, yaxis='y2'))

fig.update_layout(
    title='HSI vs CPI Indices (Levels)',
    xaxis_title='Date',
    yaxis=dict(title='HSI Close'),
    yaxis2=dict(title='CPI Index', overlaying='y', side='right'),
    hovermode='x unified',
    template='plotly_white',
    dragmode='pan',
)
fig.show(config={"scrollZoom": True, "doubleClick": "reset+autosize"})

z_cols = ['HSI_Close'] + cpi_index_cols
z = df[['Date'] + z_cols].copy()
for col in z_cols:
    z[col] = (z[col] - z[col].mean()) / z[col].std()

z_long = z.melt(id_vars='Date', var_name='Series', value_name='zscore')
fig2 = px.line(
    z_long,
    x='Date',
    y='zscore',
    color='Series',
    title='Standardized Co-movement: HSI and CPI Indices',
)
fig2.update_layout(template='plotly_white', yaxis_title='z-score', dragmode='pan')
fig2.show(config={"scrollZoom": True, "doubleClick": "reset+autosize"})


## 7) Contemporaneous relationships

In [ ]:
cpi_metric_cols = [
    c for c in df.columns
    if c.endswith('_Index') or c.endswith('_YoY_Pct') or c.endswith('_MoM_Pct')
]

corr_cols = [c for c in [
    'HSI_Close', 'HSI_Monthly_Return', 'HSI_YoY_Return', 'HSI_DailyRet_Std',
    *cpi_metric_cols,
] if c in df.columns]

corr_matrix = df[corr_cols].corr()
display(corr_matrix)

fig = px.imshow(
    corr_matrix,
    color_continuous_scale='RdBu_r',
    zmin=-1,
    zmax=1,
    title='Correlation Matrix (HSI and CPI Variants)',
)
fig.update_layout(template='plotly_white', dragmode='pan')
fig.show(config={"scrollZoom": True, "doubleClick": "reset+autosize"})

yoy_cols = [c for c in ['CCPI_YoY_Pct', 'CPI_A_YoY_Pct', 'CPI_B_YoY_Pct', 'CPI_C_YoY_Pct'] if c in df.columns]
if 'HSI_Monthly_Return' in df.columns and yoy_cols:
    scatter_df = df[['HSI_Monthly_Return'] + yoy_cols].dropna().melt(
        id_vars='HSI_Monthly_Return',
        var_name='CPI_Series',
        value_name='CPI_YoY_Pct',
    )
    fig2 = px.scatter(
        scatter_df,
        x='CPI_YoY_Pct',
        y='HSI_Monthly_Return',
        color='CPI_Series',
        title='HSI Monthly Return vs CPI YoY % (All CPI Variants)',
        trendline='ols',
    )
    fig2.update_layout(template='plotly_white', dragmode='pan')
    fig2.show(config={"scrollZoom": True, "doubleClick": "reset+autosize"})
else:
    print('No YoY CPI columns available for scatter comparison.')

## 8) Lead-lag analysis

Positive lag means CPI leads HSI by that many months.


In [ ]:
def lagged_corr(x: pd.Series, y: pd.Series, lags=range(-12, 13)) -> pd.DataFrame:
    rows = []
    for lag in lags:
        rows.append({'lag': lag, 'corr': x.corr(y.shift(lag))})
    return pd.DataFrame(rows)

cpi_yoy_cols = [c for c in ['CCPI_YoY_Pct', 'CPI_A_YoY_Pct', 'CPI_B_YoY_Pct', 'CPI_C_YoY_Pct'] if c in df.columns]
cpi_mom_cols = [c for c in ['CCPI_MoM_Pct', 'CPI_A_MoM_Pct', 'CPI_B_MoM_Pct', 'CPI_C_MoM_Pct'] if c in df.columns]

pairs = []
for c_col in cpi_yoy_cols:
    if {'HSI_Monthly_Return', c_col}.issubset(df.columns):
        pairs.append(('HSI_Monthly_Return', c_col))
    if {'HSI_YoY_Return', c_col}.issubset(df.columns):
        pairs.append(('HSI_YoY_Return', c_col))
for c_col in cpi_mom_cols:
    if {'HSI_Monthly_Return', c_col}.issubset(df.columns):
        pairs.append(('HSI_Monthly_Return', c_col))

for h_col, c_col in pairs:
    lc = lagged_corr(df[h_col], df[c_col])
    valid = lc.dropna(subset=['corr']).copy()

    if valid.empty:
        print(f'{h_col} vs {c_col}: no valid lagged correlations (insufficient overlapping data).')
        continue

    best_idx = valid['corr'].abs().idxmax()
    best = valid.loc[best_idx]
    print(f'{h_col} vs {c_col}: strongest |corr| at lag={int(best["lag"])} -> corr={best["corr"]:.3f}')

    fig = px.bar(
        lc,
        x='lag',
        y='corr',
        title=f'Lagged Correlation: {h_col} vs {c_col}',
        labels={'lag': 'Lag (months, + means CPI leads)', 'corr': 'Correlation'},
    )
    fig.add_hline(y=0, line_color='black', line_width=1)
    fig.update_layout(template='plotly_white', dragmode='pan')
    fig.show(config={"scrollZoom": True, "doubleClick": "reset+autosize"})

## 9) Rolling correlation stability

In [ ]:
cpi_yoy_cols = [c for c in ['CCPI_YoY_Pct', 'CPI_A_YoY_Pct', 'CPI_B_YoY_Pct', 'CPI_C_YoY_Pct'] if c in df.columns]

rows = []
if 'HSI_Monthly_Return' in df.columns and cpi_yoy_cols:
    for c_col in cpi_yoy_cols:
        for w in [12, 24, 36]:
            roll = df['HSI_Monthly_Return'].rolling(w).corr(df[c_col])
            tmp = pd.DataFrame({'Date': df['Date'], 'RollingCorr': roll})
            tmp['Window'] = f'{w}M'
            tmp['CPI_Series'] = c_col
            rows.append(tmp)

if rows:
    roll_df = pd.concat(rows, ignore_index=True)
    fig = px.line(
        roll_df,
        x='Date',
        y='RollingCorr',
        color='CPI_Series',
        line_dash='Window',
        title='Rolling Correlation: HSI Monthly Return vs CPI YoY (All CPI Variants)',
    )
    fig.add_hline(y=0, line_color='black', line_width=1)
    fig.update_layout(template='plotly_white', dragmode='pan')
    fig.show(config={"scrollZoom": True, "doubleClick": "reset+autosize"})
else:
    print('Required columns missing for rolling correlation plot.')

## 10) Regime and seasonality checks

In [ ]:
tmp = df.copy()
tmp['Year'] = tmp['Date'].dt.year
tmp['Month'] = tmp['Date'].dt.month

yoy_candidates = [c for c in ['CCPI_YoY_Pct', 'CPI_A_YoY_Pct', 'CPI_B_YoY_Pct', 'CPI_C_YoY_Pct'] if c in tmp.columns]
regime_source = 'CCPI_YoY_Pct' if 'CCPI_YoY_Pct' in yoy_candidates else (yoy_candidates[0] if yoy_candidates else None)

if regime_source is not None:
    tmp['CPI_Regime'] = pd.cut(
        tmp[regime_source],
        bins=[-np.inf, 0, 2, 4, np.inf],
        labels=['Deflation', 'Low Inflation', 'Moderate Inflation', 'High Inflation']
    )

if {'CPI_Regime', 'HSI_Monthly_Return'}.issubset(tmp.columns):
    fig = px.box(
        tmp,
        x='CPI_Regime',
        y='HSI_Monthly_Return',
        color='CPI_Regime',
        title=f'HSI Monthly Return by CPI Regime ({regime_source})',
    )
    fig.update_layout(template='plotly_white', dragmode='pan', showlegend=False)
    fig.show(config={"scrollZoom": True, "doubleClick": "reset+autosize"})

fig2 = px.box(
    tmp,
    x='Month',
    y='HSI_Monthly_Return',
    title='HSI Monthly Return Seasonality',
)
fig2.update_layout(template='plotly_white', dragmode='pan')
fig2.show(config={"scrollZoom": True, "doubleClick": "reset+autosize"})

## 11) OLS regression diagnostics

This is descriptive, not causal proof.


In [ ]:
cpi_groups = ['CCPI', 'CPI_A', 'CPI_B', 'CPI_C']
model_rows = []
model_store = {}

for grp in cpi_groups:
    yoy_col = f'{grp}_YoY_Pct'
    mom_col = f'{grp}_MoM_Pct'
    required = ['HSI_Monthly_Return', yoy_col]

    if not set(required).issubset(df.columns):
        continue

    X_cols = [c for c in [yoy_col, mom_col] if c in df.columns]
    reg_df = df[['HSI_Monthly_Return'] + X_cols].dropna().copy()
    if len(reg_df) < 36:
        continue

    y = reg_df['HSI_Monthly_Return']
    X = sm.add_constant(reg_df[X_cols])
    model = sm.OLS(y, X).fit(cov_type='HC3')
    model_store[grp] = (model, reg_df, X_cols)

    model_rows.append({
        'CPI_Group': grp,
        'n_obs': int(model.nobs),
        'r2': float(model.rsquared),
        'adj_r2': float(model.rsquared_adj),
        'coef_const': float(model.params.get('const', np.nan)),
        f'coef_{yoy_col}': float(model.params.get(yoy_col, np.nan)),
        f'coef_{mom_col}': float(model.params.get(mom_col, np.nan)),
        f'p_{yoy_col}': float(model.pvalues.get(yoy_col, np.nan)),
        f'p_{mom_col}': float(model.pvalues.get(mom_col, np.nan)),
    })

if model_rows:
    model_summary = pd.DataFrame(model_rows).sort_values('CPI_Group')
    display(model_summary)

    # Diagnostics for CCPI if available, else first available model
    diag_grp = 'CCPI' if 'CCPI' in model_store else list(model_store.keys())[0]
    model, reg_df, X_cols = model_store[diag_grp]
    reg_df = reg_df.copy()
    reg_df['fitted'] = model.fittedvalues
    reg_df['resid'] = model.resid

    fig = px.scatter(
        reg_df,
        x='fitted',
        y='resid',
        title=f'Residuals vs Fitted ({diag_grp})',
        opacity=0.7,
    )
    fig.add_hline(y=0, line_color='black', line_width=1)
    fig.update_layout(template='plotly_white', dragmode='pan')
    fig.show(config={"scrollZoom": True, "doubleClick": "reset+autosize"})

    qq = stats.probplot(reg_df['resid'], dist='norm')
    theo_q = qq[0][0]
    ordered = qq[0][1]
    slope, intercept = qq[1][0], qq[1][1]

    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(x=theo_q, y=ordered, mode='markers', name='Residual quantiles'))
    fig2.add_trace(go.Scatter(x=theo_q, y=slope * theo_q + intercept, mode='lines', name='Reference line'))
    fig2.update_layout(
        title=f'Residual Q-Q Plot ({diag_grp})',
        xaxis_title='Theoretical quantiles',
        yaxis_title='Ordered values',
        template='plotly_white',
        dragmode='pan',
    )
    fig2.show(config={"scrollZoom": True, "doubleClick": "reset+autosize"})
else:
    print('No valid regression setup found across CPI variants.')

## 12) Granger causality tests

Tests predictive content (not true causality). We test both directions.


In [ ]:
def adf_pvalue(series):
    s = pd.Series(series).dropna()
    if len(s) < 20:
        return np.nan
    return adfuller(s, autolag='AIC')[1]

def make_stationary(series, alpha=0.05, max_diff=3):
    s = pd.Series(series).dropna()
    d = 0
    p = adf_pvalue(s)
    while pd.notna(p) and p > alpha and d < max_diff:
        s = s.diff().dropna()
        d += 1
        p = adf_pvalue(s)
    return s, d, p

hsi_col = 'HSI_Monthly_Return'
cpi_yoy_cols = [c for c in ['CCPI_YoY_Pct', 'CPI_A_YoY_Pct', 'CPI_B_YoY_Pct', 'CPI_C_YoY_Pct'] if c in df.columns]

if hsi_col not in df.columns or not cpi_yoy_cols:
    print('Required columns missing for Granger test.')
else:
    adf_rows = []
    granger_rows = []
    max_lag = 12

    for c_col in cpi_yoy_cols:
        gc = df[[hsi_col, c_col]].dropna().copy()
        if len(gc) <= 48:
            adf_rows.append({'Series': c_col, 'HSI_diff_order': np.nan, 'HSI_adf_p': np.nan, 'CPI_diff_order': np.nan, 'CPI_adf_p': np.nan, 'obs': len(gc)})
            continue

        hsi_s, hsi_d, hsi_p = make_stationary(gc[hsi_col])
        cpi_s, cpi_d, cpi_p = make_stationary(gc[c_col])

        adf_rows.append({
            'Series': c_col,
            'HSI_diff_order': hsi_d,
            'HSI_adf_p': hsi_p,
            'CPI_diff_order': cpi_d,
            'CPI_adf_p': cpi_p,
            'obs': len(gc),
        })

        st = pd.concat([hsi_s.rename(hsi_col), cpi_s.rename(c_col)], axis=1).dropna()
        if len(st) <= (max_lag + 15):
            continue

        with warnings.catch_warnings():
            warnings.simplefilter('ignore', FutureWarning)
            res1 = grangercausalitytests(st[[hsi_col, c_col]], maxlag=max_lag, verbose=False)
            res2 = grangercausalitytests(st[[c_col, hsi_col]], maxlag=max_lag, verbose=False)

        pvals1 = {lag: out[0]['ssr_ftest'][1] for lag, out in res1.items()}
        pvals2 = {lag: out[0]['ssr_ftest'][1] for lag, out in res2.items()}

        granger_rows.append({
            'Series': c_col,
            'Direction': f'{c_col} -> {hsi_col}',
            'Best_Lag': min(pvals1, key=pvals1.get),
            'Min_p_value': float(min(pvals1.values())),
        })
        granger_rows.append({
            'Series': c_col,
            'Direction': f'{hsi_col} -> {c_col}',
            'Best_Lag': min(pvals2, key=pvals2.get),
            'Min_p_value': float(min(pvals2.values())),
        })

    adf_summary = pd.DataFrame(adf_rows)
    granger_summary = pd.DataFrame(granger_rows)

    display(adf_summary)
    display(granger_summary.sort_values('Min_p_value'))

    if not granger_summary.empty:
        fig = px.bar(
            granger_summary.sort_values('Min_p_value'),
            x='Series',
            y='Min_p_value',
            color='Direction',
            barmode='group',
            hover_data=['Best_Lag'],
            title='Granger Causality by CPI Variant (Min p-value across lags)',
        )
        fig.add_hline(y=0.05, line_dash='dash', line_color='red')
        fig.update_layout(template='plotly_white', dragmode='pan')
        fig.show(config={"scrollZoom": True, "doubleClick": "reset+autosize"})

## 13) Key findings template

Run this final cell after all analyses to get quick machine-generated highlights.


In [ ]:
highlights = []

if {'HSI_Monthly_Return', 'CPI_YoY_Pct'}.issubset(df.columns):
    corr_val = df['HSI_Monthly_Return'].corr(df['CPI_YoY_Pct'])
    highlights.append(f'Contemporaneous corr(HSI monthly return, CPI YoY) = {corr_val:.3f}')

    lc = pd.DataFrame({'lag': range(-12, 13)})
    lc['corr'] = [df['HSI_Monthly_Return'].corr(df['CPI_YoY_Pct'].shift(l)) for l in lc['lag']]
    valid = lc.dropna(subset=['corr']).copy()
    if not valid.empty:
        best = valid.loc[valid['corr'].abs().idxmax()]
        direction = 'CPI leads HSI' if best['lag'] > 0 else ('HSI leads CPI' if best['lag'] < 0 else 'same month')
        highlights.append(f'Strongest lagged |corr| at lag={int(best["lag"])} ({direction}), corr={best["corr"]:.3f}')
    else:
        highlights.append('No valid lagged correlation values (insufficient overlap after missing-value handling).')

if highlights:
    print('Quick highlights:')
    for i, h in enumerate(highlights, 1):
        print(f'{i}. {h}')
else:
    print('No highlights available yet; check earlier data preparation cells.')


## 14) Findings and conclusions (from current notebook outputs)

### Good findings
- The analysis window is long enough for monthly inference after merge: 312 observations (2000-01 to 2025-12).
- Data completeness is strong in the merged frame: `CPI_Index`, `CPI_YoY_Pct`, and `CPI_MoM_Pct` show 0 missing values; only expected early missing values appear for `HSI_Monthly_Return` (first row) and `HSI_YoY_Return` (first 12 rows).
- There is a clear long-run co-trending relationship in levels: `corr(HSI_Close, CPI_Index) ≈ 0.614` and `corr(HSI_Close, CPI_YoY_Pct) ≈ 0.567`.
- Lag analysis suggests stronger lead-lag structure than same-month association: for CCPI YoY, strongest absolute lagged correlation occurs around lag `-10` (HSI leads CPI in this setup), with `corr ≈ 0.143`.
- Granger results indicate statistically significant predictive content in several directions, especially `HSI_Monthly_Return -> CPI_*_YoY_Pct` at long lags (best p-values as low as `~4.3e-07`).

### Bad findings / limitations
- Same-month return-inflation relation is weak: `corr(HSI_Monthly_Return, CPI_YoY_Pct) = -0.047` (economically small).
- Explanatory power of linear OLS models is low (`R^2` about `0.016` to `0.029` across CPI groups), so CPI features alone explain only a small share of monthly HSI return variation.
- In OLS, YoY CPI terms are mostly not significant, while MoM terms are more often significant; this can indicate short-term noise sensitivity and model instability.
- Level correlations likely include shared trend effects (potential spurious-correlation risk), so they should not be interpreted as causal evidence.
- Granger significance appears stronger from HSI to CPI than CPI to HSI in this run; interpretation should be cautious because Granger tests are predictive, not structural causality.
- Output consistency check needed: the CPI-cleaning output shown in the notebook reports dates before 2000, while merged analysis is 2000 onward; this should be re-run and verified to ensure cleaning-stage filtering is reflected in outputs.

### Conclusions
- There is no strong evidence of a robust same-month CPI-to-HSI-return relationship.
- The dominant signal is temporal: lead-lag structure exists, with HSI often leading CPI metrics in this specification.
- CPI is useful as part of a broader feature set, but insufficient as a standalone driver for HSI monthly returns.
- For stronger inference, next iterations should add macro controls (rates, FX, liquidity/global risk proxies), sub-period robustness checks, and out-of-sample validation.


## 15) Additional Tests Requested (Added)

The following were **already implemented earlier** and are not duplicated here:
- Contemporaneous correlation and OLS (Section 7/11)
- Lag-based exploratory relationships (Section 8)
- Granger causality (Section 12)

New sections below add the missing tests and stronger variants.


In [ ]:
# Shared variable selection for added tests
ret_col = 'HSI_Monthly_Return'
close_col = 'HSI_Close'

# Prefer CCPI as primary inflation series; fallback to general CPI columns if needed
primary_yoy_candidates = ['CCPI_YoY_Pct', 'CPI_YoY_Pct']
primary_mom_candidates = ['CCPI_MoM_Pct', 'CPI_MoM_Pct']
primary_idx_candidates = ['CCPI_Index', 'CPI_Index']

primary_yoy = next((c for c in primary_yoy_candidates if c in df.columns), None)
primary_mom = next((c for c in primary_mom_candidates if c in df.columns), None)
primary_idx = next((c for c in primary_idx_candidates if c in df.columns), None)

print('Selected primary CPI columns:')
print({'YoY': primary_yoy, 'MoM': primary_mom, 'Index': primary_idx})


## 16) Lagged Predictability (OLS, lags 1-12)

Model: `HSI_return_t = alpha + beta * CPI_t-k` for `k = 1..12`.
Compared using adjusted `R^2` and p-values.


In [ ]:
lag_rows = []

if primary_yoy is None or ret_col not in df.columns:
    print('Required columns missing for lagged predictability test.')
else:
    for k in range(1, 13):
        test_df = df[[ret_col, primary_yoy]].copy()
        test_df[f'{primary_yoy}_lag{k}'] = test_df[primary_yoy].shift(k)
        test_df = test_df[[ret_col, f'{primary_yoy}_lag{k}']].dropna()

        if len(test_df) < 36:
            continue

        y = test_df[ret_col]
        X = sm.add_constant(test_df[[f'{primary_yoy}_lag{k}']])
        model = sm.OLS(y, X).fit(cov_type='HC3')

        lag_rows.append({
            'lag': k,
            'n_obs': int(model.nobs),
            'adj_r2': float(model.rsquared_adj),
            'coef_beta': float(model.params[f'{primary_yoy}_lag{k}']),
            'p_value': float(model.pvalues[f'{primary_yoy}_lag{k}']),
        })

    lag_ols_summary = pd.DataFrame(lag_rows).sort_values('lag') if lag_rows else pd.DataFrame()
    display(lag_ols_summary)

    if not lag_ols_summary.empty:
        best_row = lag_ols_summary.loc[lag_ols_summary['adj_r2'].idxmax()]
        print(f"Best lag by adjusted R^2: lag={int(best_row['lag'])}, adj_R^2={best_row['adj_r2']:.4f}, p={best_row['p_value']:.4g}")

        fig = px.line(
            lag_ols_summary,
            x='lag',
            y='adj_r2',
            markers=True,
            title=f'Lagged OLS Predictability: {ret_col} ~ {primary_yoy}(t-k)'
        )
        fig.update_layout(template='plotly_white', dragmode='pan')
        fig.show(config={'scrollZoom': True, 'doubleClick': 'reset+autosize'})


## 17) Cross-Correlation Function (CCF)

Positive lag = CPI leads HSI returns by that many months.
Includes approximate 95% confidence bounds `±1.96/sqrt(N)`.


In [ ]:
if primary_yoy is None or ret_col not in df.columns:
    print('Required columns missing for CCF test.')
else:
    ccf_rows = []
    max_lag = 12
    for lag in range(0, max_lag + 1):
        tmp = df[[ret_col, primary_yoy]].copy()
        tmp['cpi_lag'] = tmp[primary_yoy].shift(lag)
        tmp = tmp[[ret_col, 'cpi_lag']].dropna()
        if len(tmp) < 20:
            corr = np.nan
            n_eff = 0
        else:
            corr = tmp[ret_col].corr(tmp['cpi_lag'])
            n_eff = len(tmp)
        ccf_rows.append({'lag': lag, 'ccf': corr, 'n_eff': n_eff})

    ccf_df = pd.DataFrame(ccf_rows)
    n_used = int(ccf_df['n_eff'].max()) if ccf_df['n_eff'].max() > 0 else 0
    conf = 1.96 / np.sqrt(n_used) if n_used > 0 else np.nan

    display(ccf_df)

    fig = px.bar(ccf_df, x='lag', y='ccf', title=f'CCF: {primary_yoy} leading {ret_col}')
    if pd.notna(conf):
        fig.add_hline(y=conf, line_dash='dash', line_color='red')
        fig.add_hline(y=-conf, line_dash='dash', line_color='red')
    fig.add_hline(y=0, line_color='black')
    fig.update_layout(template='plotly_white', dragmode='pan')
    fig.show(config={'scrollZoom': True, 'doubleClick': 'reset+autosize'})


## 18) Inflation & Volatility (Risk Channel)

A) Rolling 12M HSI volatility vs CPI YoY

B) Optional GARCH-X: conditional volatility with CPI exogenous input (runs only if `arch` package is available).


In [ ]:
if primary_yoy is None or ret_col not in df.columns:
    print('Required columns missing for volatility channel test.')
else:
    vol_df = df[['Date', ret_col, primary_yoy]].copy()
    vol_df['HSI_Roll12M_Vol'] = vol_df[ret_col].rolling(12).std() * np.sqrt(12)
    vol_df = vol_df.dropna().reset_index(drop=True)

    vol_corr_pearson = vol_df['HSI_Roll12M_Vol'].corr(vol_df[primary_yoy], method='pearson')
    vol_corr_spearman = vol_df['HSI_Roll12M_Vol'].corr(vol_df[primary_yoy], method='spearman')

    print(f'Pearson corr(12M vol, CPI YoY) = {vol_corr_pearson:.4f}')
    print(f'Spearman corr(12M vol, CPI YoY) = {vol_corr_spearman:.4f}')

    y = vol_df['HSI_Roll12M_Vol']
    X = sm.add_constant(vol_df[[primary_yoy]])
    vol_model = sm.OLS(y, X).fit(cov_type='HC3')
    print(vol_model.summary())

    fig = px.scatter(
        vol_df,
        x=primary_yoy,
        y='HSI_Roll12M_Vol',
        trendline='ols',
        title='Rolling 12M HSI Volatility vs CPI YoY'
    )
    fig.update_layout(template='plotly_white', dragmode='pan')
    fig.show(config={'scrollZoom': True, 'doubleClick': 'reset+autosize'})


In [ ]:
# Optional: GARCH-X using CPI as exogenous variable in variance model
try:
    from arch import arch_model

    if primary_yoy is None or ret_col not in df.columns:
        print('Required columns missing for GARCH-X test.')
    else:
        gdf = df[[ret_col, primary_yoy]].dropna().copy()
        gdf['ret_pct'] = gdf[ret_col] * 100.0

        # Mean: constant; Volatility: GARCH(1,1); Exogenous regressor in mean equation
        # (arch package does not directly place exog in variance in this simple API path)
        garchx = arch_model(
            gdf['ret_pct'],
            x=gdf[[primary_yoy]],
            mean='ARX',
            lags=1,
            vol='GARCH',
            p=1,
            q=1,
            dist='normal'
        )
        garchx_res = garchx.fit(disp='off')
        print(garchx_res.summary())
except Exception as e:
    print('GARCH-X skipped (optional dependency or model issue):', e)


## 19) Inflation Regime Inference: ANOVA, Levene, Structural Break (Chow)


In [ ]:
regime_df = df[['Date', ret_col, primary_yoy]].dropna().copy() if primary_yoy is not None else pd.DataFrame()

if regime_df.empty:
    print('Required columns missing for regime tests.')
else:
    regime_df['Regime'] = pd.cut(
        regime_df[primary_yoy],
        bins=[-np.inf, 2, 4, np.inf],
        labels=['Low(<2)', 'Moderate(2-4)', 'High(>4)']
    )

    groups = [g[ret_col].dropna().values for _, g in regime_df.groupby('Regime') if len(g) > 5]
    if len(groups) >= 2:
        anova_stat, anova_p = stats.f_oneway(*groups)
        lev_stat, lev_p = stats.levene(*groups, center='median')
        print(f'ANOVA (mean returns across regimes): F={anova_stat:.4f}, p={anova_p:.4g}')
        print(f"Levene (variance equality across regimes): W={lev_stat:.4f}, p={lev_p:.4g}")
    else:
        print('Insufficient observations per regime for ANOVA/Levene.')

    fig = px.box(regime_df, x='Regime', y=ret_col, color='Regime', title='HSI Returns Across Inflation Regimes')
    fig.update_layout(template='plotly_white', dragmode='pan', showlegend=False)
    fig.show(config={'scrollZoom': True, 'doubleClick': 'reset+autosize'})

    def chow_test(data, y_col, x_col, break_date):
        d = data[['Date', y_col, x_col]].dropna().copy()
        d1 = d[d['Date'] < pd.Timestamp(break_date)].copy()
        d2 = d[d['Date'] >= pd.Timestamp(break_date)].copy()

        if len(d1) < 30 or len(d2) < 30:
            return {'break_date': break_date, 'error': 'insufficient sample split'}

        def fit_rss(sub):
            y = sub[y_col]
            X = sm.add_constant(sub[[x_col]])
            m = sm.OLS(y, X).fit()
            return float((m.resid ** 2).sum()), int(m.df_model + 1)

        rss_pooled, k = fit_rss(d)
        rss1, _ = fit_rss(d1)
        rss2, _ = fit_rss(d2)

        n1, n2 = len(d1), len(d2)
        denom_df = n1 + n2 - 2 * k
        if denom_df <= 0:
            return {'break_date': break_date, 'error': 'invalid degrees of freedom'}

        f_stat = ((rss_pooled - (rss1 + rss2)) / k) / ((rss1 + rss2) / denom_df)
        p_val = 1 - stats.f.cdf(f_stat, k, denom_df)
        return {
            'break_date': break_date,
            'n_pre': n1,
            'n_post': n2,
            'F_stat': f_stat,
            'p_value': p_val,
        }

    chow_rows = [
        chow_test(regime_df, ret_col, primary_yoy, '2008-09-01'),
        chow_test(regime_df, ret_col, primary_yoy, '2020-03-01'),
    ]
    display(pd.DataFrame(chow_rows))


## 20) Inflation Hedge Hypothesis and Unexpected Inflation

A) Ex-post hedge: `HSI_return ~ inflation`

B) Unexpected inflation model:
1. Fit AR(12) on inflation (expected component)
2. Unexpected inflation = residual
3. Regress returns on unexpected inflation


In [ ]:
if primary_yoy is None or ret_col not in df.columns:
    print('Required columns missing for hedge tests.')
else:
    hdf = df[[ret_col, primary_yoy]].dropna().copy()

    # A) Ex-post hedge
    y = hdf[ret_col]
    X = sm.add_constant(hdf[[primary_yoy]])
    hedge_model = sm.OLS(y, X).fit(cov_type='HC3')
    beta = hedge_model.params.get(primary_yoy, np.nan)
    print(hedge_model.summary())

    if pd.notna(beta):
        if beta >= 0.8:
            print(f'Ex-post hedge read: beta={beta:.3f} (closer to 1, stronger hedge signal).')
        elif beta > 0:
            print(f'Ex-post hedge read: beta={beta:.3f} (positive but weak hedge signal).')
        else:
            print(f'Ex-post hedge read: beta={beta:.3f} (negative, poor hedge signal).')

    # B) Unexpected inflation via AR(12)
    inf = hdf[primary_yoy].copy()
    ar_model = sm.tsa.AutoReg(inf, lags=12, old_names=False).fit()
    expected_inf = ar_model.fittedvalues.reindex(inf.index)
    unexpected_inf = inf - expected_inf

    u_df = pd.DataFrame({
        ret_col: hdf[ret_col],
        'unexpected_inflation': unexpected_inf,
    }).dropna()

    if len(u_df) >= 36:
        y2 = u_df[ret_col]
        X2 = sm.add_constant(u_df[['unexpected_inflation']])
        u_model = sm.OLS(y2, X2).fit(cov_type='HC3')
        print(u_model.summary())
    else:
        print('Insufficient observations for unexpected inflation regression.')


## 21) CPI Sub-index Multivariate Regression (A/B/C)

Model: `HSI_return ~ CPI_A + CPI_B + CPI_C` (YoY and MoM variants)


In [ ]:
sub_yoy = [c for c in ['CPI_A_YoY_Pct', 'CPI_B_YoY_Pct', 'CPI_C_YoY_Pct'] if c in df.columns]
sub_mom = [c for c in ['CPI_A_MoM_Pct', 'CPI_B_MoM_Pct', 'CPI_C_MoM_Pct'] if c in df.columns]

if ret_col not in df.columns or len(sub_yoy) < 3:
    print('Required CPI sub-index columns missing for multivariate regression.')
else:
    mdf = df[[ret_col] + sub_yoy + sub_mom].dropna().copy()

    X_cols = sub_yoy + sub_mom
    y = mdf[ret_col]
    X = sm.add_constant(mdf[X_cols])
    m_model = sm.OLS(y, X).fit(cov_type='HC3')
    print(m_model.summary())

    coef_table = pd.DataFrame({
        'coef': m_model.params,
        'p_value': m_model.pvalues,
    })
    display(coef_table)


## 22) Inflation Momentum and Asymmetry

Momentum tests use `Δ CPI_YoY` and acceleration `Δ² CPI_YoY`.
Asymmetry compares returns in rising vs falling inflation months.


In [ ]:
if primary_yoy is None or ret_col not in df.columns:
    print('Required columns missing for momentum/asymmetry tests.')
else:
    adf = df[['Date', ret_col, primary_yoy]].copy()
    adf['dCPI_YoY'] = adf[primary_yoy].diff(1)
    adf['ddCPI_YoY'] = adf['dCPI_YoY'].diff(1)

    # Momentum regression
    mom_df = adf[[ret_col, 'dCPI_YoY', 'ddCPI_YoY']].dropna().copy()
    if len(mom_df) >= 36:
        y = mom_df[ret_col]
        X = sm.add_constant(mom_df[['dCPI_YoY', 'ddCPI_YoY']])
        mom_model = sm.OLS(y, X).fit(cov_type='HC3')
        print(mom_model.summary())
    else:
        print('Insufficient data for momentum regression.')

    # Asymmetry test
    adf['inflation_direction'] = np.where(adf['dCPI_YoY'] > 0, 'Rising', 'Falling_or_Flat')
    asym = adf[[ret_col, 'inflation_direction']].dropna()

    rising = asym.loc[asym['inflation_direction'] == 'Rising', ret_col]
    falling = asym.loc[asym['inflation_direction'] == 'Falling_or_Flat', ret_col]

    if len(rising) > 10 and len(falling) > 10:
        t_stat, t_p = stats.ttest_ind(rising, falling, equal_var=False, nan_policy='omit')
        print(f'Asymmetry t-test (Rising vs Falling/Flat): t={t_stat:.4f}, p={t_p:.4g}')
        print(f'Mean return Rising={rising.mean():.4f}, Falling/Flat={falling.mean():.4f}')

        fig = px.box(asym, x='inflation_direction', y=ret_col, color='inflation_direction', title='Asymmetry: HSI Returns by Inflation Direction')
        fig.update_layout(template='plotly_white', dragmode='pan', showlegend=False)
        fig.show(config={'scrollZoom': True, 'doubleClick': 'reset+autosize'})
    else:
        print('Insufficient sample for asymmetry t-test.')


## 23) Event Study: Inflation Spikes vs Drops

Events are based on top/bottom 10 monthly changes in CPI YoY (`Δ CPI_YoY`).
Outcome: forward HSI returns over 1M and 3M horizons.


In [ ]:
if primary_yoy is None or close_col not in df.columns:
    print('Required columns missing for event study.')
else:
    ev = df[['Date', close_col, primary_yoy]].copy().sort_values('Date').reset_index(drop=True)
    ev['dCPI_YoY'] = ev[primary_yoy].diff(1)

    ev['Fwd_1M'] = ev[close_col].shift(-1) / ev[close_col] - 1
    ev['Fwd_3M'] = ev[close_col].shift(-3) / ev[close_col] - 1

    base = ev.dropna(subset=['dCPI_YoY', 'Fwd_1M', 'Fwd_3M']).copy()
    if len(base) < 30:
        print('Insufficient observations for event study.')
    else:
        spikes = base.nlargest(10, 'dCPI_YoY').copy()
        drops = base.nsmallest(10, 'dCPI_YoY').copy()

        spikes['Event'] = 'Inflation Spike'
        drops['Event'] = 'Inflation Drop'
        events = pd.concat([spikes, drops], ignore_index=True)

        summary = events.groupby('Event')[['Fwd_1M', 'Fwd_3M']].agg(['mean', 'median', 'std', 'count'])
        display(summary)

        t1 = stats.ttest_ind(spikes['Fwd_1M'], drops['Fwd_1M'], equal_var=False, nan_policy='omit')
        t3 = stats.ttest_ind(spikes['Fwd_3M'], drops['Fwd_3M'], equal_var=False, nan_policy='omit')
        print(f'1M forward return difference t-test: t={t1.statistic:.4f}, p={t1.pvalue:.4g}')
        print(f'3M forward return difference t-test: t={t3.statistic:.4f}, p={t3.pvalue:.4g}')

        fig = px.box(
            events.melt(id_vars=['Event'], value_vars=['Fwd_1M', 'Fwd_3M'], var_name='Horizon', value_name='Forward_Return'),
            x='Event',
            y='Forward_Return',
            color='Horizon',
            title='Event Study: Forward HSI Returns after Inflation Spikes vs Drops'
        )
        fig.update_layout(template='plotly_white', dragmode='pan')
        fig.show(config={'scrollZoom': True, 'doubleClick': 'reset+autosize'})


## 24) Cointegration Tests (Long-run Relationship)

Because HSI and CPI levels trend over time, we test for cointegration using:
- Engle-Granger two-step test
- Johansen test


In [ ]:
try:
    from statsmodels.tsa.stattools import coint
    from statsmodels.tsa.vector_ar.vecm import coint_johansen

    if primary_idx is None or close_col not in df.columns:
        print('Required level columns missing for cointegration tests.')
    else:
        cdf = df[[close_col, primary_idx]].dropna().copy()
        cdf = cdf[(cdf[close_col] > 0) & (cdf[primary_idx] > 0)].copy()

        y = np.log(cdf[close_col])
        x = np.log(cdf[primary_idx])

        eg_stat, eg_p, eg_crit = coint(y, x)
        print('Engle-Granger test (log levels):')
        print({'test_stat': float(eg_stat), 'p_value': float(eg_p), 'crit_vals': eg_crit})

        j_input = np.column_stack([y.values, x.values])
        joh = coint_johansen(j_input, det_order=0, k_ar_diff=1)
        joh_summary = pd.DataFrame({
            'rank': [0, 1],
            'trace_stat': joh.lr1,
            'trace_95pct_crit': joh.cvt[:, 1],
            'maxeig_stat': joh.lr2,
            'maxeig_95pct_crit': joh.cvm[:, 1],
        })
        print('Johansen test (log levels):')
        display(joh_summary)
except Exception as e:
    print('Cointegration tests skipped due to dependency/runtime issue:', e)


## 25) Updated Findings and Conclusions (Auto-Summary)

Run the next code cell after Sections 15-24 to automatically produce a concise findings report.

Interpretation guide:
- `p < 0.05`: statistically significant
- Higher adjusted `R^2`: stronger in-sample explanatory power
- For hedge beta (`HSI_return ~ inflation`):
  - `beta ~ 1`: strong hedge
  - `0 < beta < 1`: partial/weak hedge
  - `beta <= 0`: poor hedge

This summary consolidates:
- Lagged predictability (best lag)
- CCF lead-lag signal
- Volatility channel strength
- Regime ANOVA/Levene + Chow breaks
- Hedge and unexpected inflation regressions
- CPI A/B/C multivariate importance
- Event-study spike/drop differences
- Cointegration evidence


In [ ]:
# Auto-summary findings from Sections 15-24
from math import isnan

summary_lines = []

def fmt_num(x, d=4):
    try:
        if x is None or (isinstance(x, float) and np.isnan(x)):
            return 'NA'
        return f'{float(x):.{d}f}'
    except Exception:
        return str(x)

# 1) Lagged OLS
if 'lag_ols_summary' in globals() and isinstance(lag_ols_summary, pd.DataFrame) and not lag_ols_summary.empty:
    best = lag_ols_summary.loc[lag_ols_summary['adj_r2'].idxmax()]
    sig = 'significant' if best['p_value'] < 0.05 else 'not significant'
    summary_lines.append(
        f"Lagged predictability: best lag = {int(best['lag'])}, adj_R^2 = {best['adj_r2']:.4f}, beta p = {best['p_value']:.4g} ({sig})."
    )
else:
    summary_lines.append('Lagged predictability: not available (run Section 16).')

# 2) CCF
if 'ccf_df' in globals() and isinstance(ccf_df, pd.DataFrame) and not ccf_df.empty:
    ctmp = ccf_df.dropna(subset=['ccf']).copy()
    if not ctmp.empty:
        b = ctmp.loc[ctmp['ccf'].abs().idxmax()]
        summary_lines.append(
            f"CCF: strongest absolute correlation at lag {int(b['lag'])} with ccf = {b['ccf']:.4f} (positive lag means CPI leads)."
        )
    else:
        summary_lines.append('CCF: computed but no valid correlation values.')
else:
    summary_lines.append('CCF: not available (run Section 17).')

# 3) Volatility channel
if 'vol_corr_pearson' in globals() and 'vol_corr_spearman' in globals():
    summary_lines.append(
        f"Volatility channel: corr(12M vol, CPI YoY) Pearson = {fmt_num(vol_corr_pearson)}, Spearman = {fmt_num(vol_corr_spearman)}."
    )
else:
    summary_lines.append('Volatility channel: not available (run Section 18).')

# 4) Regime tests
if 'anova_p' in globals() and 'lev_p' in globals():
    a_txt = 'significant' if anova_p < 0.05 else 'not significant'
    l_txt = 'significant' if lev_p < 0.05 else 'not significant'
    summary_lines.append(
        f"Regime tests: ANOVA p = {fmt_num(anova_p)} ({a_txt}); Levene p = {fmt_num(lev_p)} ({l_txt})."
    )
else:
    summary_lines.append('Regime tests: not available (run Section 19).')

if 'chow_rows' in globals() and isinstance(chow_rows, list) and len(chow_rows) > 0:
    chow_bits = []
    for r in chow_rows:
        if isinstance(r, dict) and 'p_value' in r:
            tag = 'break' if r['p_value'] < 0.05 else 'no break'
            chow_bits.append(f"{r.get('break_date','?')}: p={fmt_num(r['p_value'])} ({tag})")
    if chow_bits:
        summary_lines.append('Chow tests: ' + '; '.join(chow_bits) + '.')

# 5) Hedge + unexpected inflation
if 'beta' in globals():
    if beta >= 0.8:
        htxt = 'strong hedge-like'
    elif beta > 0:
        htxt = 'partial/weak hedge'
    else:
        htxt = 'poor hedge'
    summary_lines.append(f"Hedge beta: {fmt_num(beta)} ({htxt}).")
else:
    summary_lines.append('Hedge beta: not available (run Section 20).')

if 'u_model' in globals():
    p_u = u_model.pvalues.get('unexpected_inflation', np.nan)
    txt = 'significant' if pd.notna(p_u) and p_u < 0.05 else 'not significant'
    summary_lines.append(f"Unexpected inflation effect: p = {fmt_num(p_u)} ({txt}).")

# 6) CPI sub-index multivariate
if 'm_model' in globals():
    sig_terms = [k for k, v in m_model.pvalues.items() if k != 'const' and pd.notna(v) and v < 0.05]
    if sig_terms:
        summary_lines.append('CPI sub-index multivariate: significant terms -> ' + ', '.join(sig_terms) + '.')
    else:
        summary_lines.append('CPI sub-index multivariate: no CPI term significant at 5%.')
else:
    summary_lines.append('CPI sub-index multivariate: not available (run Section 21).')

# 7) Event study
if 't1' in globals() and 't3' in globals():
    e1 = 'significant' if t1.pvalue < 0.05 else 'not significant'
    e3 = 'significant' if t3.pvalue < 0.05 else 'not significant'
    summary_lines.append(
        f"Event study (spikes vs drops): 1M p={fmt_num(t1.pvalue)} ({e1}), 3M p={fmt_num(t3.pvalue)} ({e3})."
    )
else:
    summary_lines.append('Event study: not available (run Section 23).')

# 8) Cointegration
if 'eg_p' in globals():
    eg_txt = 'cointegration evidence' if eg_p < 0.05 else 'no strong cointegration evidence'
    summary_lines.append(f"Engle-Granger: p = {fmt_num(eg_p)} ({eg_txt}).")
else:
    summary_lines.append('Cointegration: not available (run Section 24).')

if 'joh_summary' in globals() and isinstance(joh_summary, pd.DataFrame) and not joh_summary.empty:
    row0 = joh_summary.iloc[0]
    tr = 'pass' if row0['trace_stat'] > row0['trace_95pct_crit'] else 'fail'
    me = 'pass' if row0['maxeig_stat'] > row0['maxeig_95pct_crit'] else 'fail'
    summary_lines.append(
        f"Johansen rank-0 check: trace={tr}, max-eigen={me} (vs 95% critical values)."
    )

print('Updated Findings (Auto):')
for i, line in enumerate(summary_lines, 1):
    print(f'{i}. {line}')


### Standalone OLS Summary: HSI vs CPI YoY

Replicates the single-regressor `statsmodels` output format shown in the screenshot for the HSI specification:

`HSI_Monthly_Return ~ CCPI_YoY_Pct`


In [ ]:
if primary_yoy is None or ret_col not in df.columns:
    print('Required columns missing for standalone OLS summary.')
else:
    ols_df = df[[ret_col, primary_yoy]].dropna().copy()
    y = ols_df[ret_col]
    X = sm.add_constant(ols_df[[primary_yoy]])
    hsi_ols_model = sm.OLS(y, X).fit(cov_type='HC3')
    print(hsi_ols_model.summary())
